# House AI — SPAR3D T4 backend

Complete Colab setup using an isolated Python 3.10 environment. **Before running Cell 4**, request access to `stabilityai/stable-point-aware-3d` on Hugging Face and create a read token.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first"
print("GPU:",torch.cuda.get_device_name(0))
print("VRAM GB:",round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))


In [ ]:
%cd /content
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
!bash /content/miniconda.sh -b -p /content/miniconda
!/content/miniconda/bin/conda create -y -n houseai python=3.10
PY='/content/miniconda/envs/houseai/bin/python'
PIP='/content/miniconda/envs/houseai/bin/pip'
print('Python 3.10 environment ready')


In [ ]:
%cd /content
!rm -rf /content/stable-point-aware-3d
!git clone --depth 1 https://github.com/Stability-AI/stable-point-aware-3d.git /content/stable-point-aware-3d
PY='/content/miniconda/envs/houseai/bin/python'
PIP='/content/miniconda/envs/houseai/bin/pip'
!$PIP install -q --upgrade pip setuptools==69.5.1 wheel
!$PIP install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
%cd /content/stable-point-aware-3d
import subprocess
subprocess.run([PIP,'install','-r','requirements.txt'],check=True)
print('SPAR3D installed successfully')


In [ ]:
from getpass import getpass
TOKEN=getpass('Paste your Hugging Face READ token: ')
import subprocess
PY='/content/miniconda/envs/houseai/bin/python'
subprocess.run([PY,'-c',"from huggingface_hub import login; import os; login(token=os.environ['HF_TOKEN'])"],env={**__import__('os').environ,'HF_TOKEN':TOKEN},check=True)
print('Hugging Face authentication saved')


In [ ]:
%cd /content
!rm -rf /content/house-ai
!git clone --depth 1 https://github.com/Rohit9605/RohitGundam-house-ai.git /content/house-ai
import os,subprocess,time
env=os.environ.copy(); env['SPAR3D_DIR']='/content/stable-point-aware-3d'
PY='/content/miniconda/envs/houseai/bin/python'
server=subprocess.Popen([PY,'/content/house-ai/local-world/server.py'],env=env)
time.sleep(3)
assert server.poll() is None, 'House AI backend exited early'
print('House AI backend started')


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
import subprocess,re,time
tunnel=subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8787','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
url=None; deadline=time.time()+60
while time.time()<deadline:
    line=tunnel.stdout.readline()
    if line: print(line,end='')
    m=re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com',line)
    if m: url=m.group(0); break
assert url,'Cloudflare tunnel did not start'
print('\nCOPY THIS URL INTO HOUSE AI:\n'+url)
print('Keep this Colab tab connected.')
